### Importing libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, FunctionTransformer, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from ydata_profiling import ProfileReport
from sklearn.metrics import accuracy_score
import scipy.stats as stats
import warnings
import pickle
warnings.filterwarnings("ignore")
%matplotlib inline

d:\Projects\ML-Projects\WineQualityE2E\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Loading data

In [2]:
df_train = pd.read_csv('D:/Projects/WineQuality/WineQualityE2E/data/raw/train.csv')
df_test = pd.read_csv('D:/Projects/WineQuality/WineQualityE2E/data/raw/test.csv')

### EDA

In [59]:
ProfileReport(df_train).to_file("diabetes_train_data_profile.html")


Summarize dataset:  32%|███▏      | 10/31 [00:04<00:09,  2.27it/s, Describe variable:systolic_bp]                      


KeyboardInterrupt: 

### Checking Correlation

In [40]:
corr = df_train.select_dtypes(include='number').corr(method='spearman')
plt.figure(figsize=(12,8))
sns.heatmap(corr, annot=True,cmap='coolwarm', fmt=".2f")
plt.title("Spearman Correlation Matrix")
plt.show()

KeyboardInterrupt: 

### Source & Target

In [3]:
x = df_train.drop(columns=['employment_status', 'diagnosed_diabetes'], axis=1)
y = df_train['diagnosed_diabetes']

### Train Test Split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state= 42)

### Preprocessing

In [5]:
log_and_scale = Pipeline(steps=[
    ('log', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
    ('scale', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('LogScale', log_and_scale,
         ['alcohol_consumption_per_week', 'physical_activity_minutes_per_week', 'triglycerides']),

        ('Scaling', StandardScaler(), [
            'sleep_hours_per_day', 'diet_score',
            'screen_time_hours_per_day', 'bmi', 'waist_to_hip_ratio',
            'systolic_bp', 'diastolic_bp', 'heart_rate',
            'cholesterol_total', 'hdl_cholesterol', 'ldl_cholesterol'
        ]),

        ('Nominal', OneHotEncoder(handle_unknown='ignore', sparse_output=False),
         ['gender', 'ethnicity']),

        ('Ordinal', OrdinalEncoder(categories=[
            ['Low', 'Lower-Middle', 'Middle', 'Upper-Middle', 'High'],
            ['Never', 'Former', 'Current'], ['No formal', 'Highschool', 'Graduate', 'Postgraduate']
        ]), ['income_level', 'smoking_status', 'education_level'])
    ],
    remainder='passthrough'
)

### Using Logistic Regression

In [8]:
LogisticRegression(
    penalty='elasticnet',
    l1_ratio=0.2,
    solver='saga',
    C=0.3,
    class_weight='balanced',
    max_iter=5000
)


LogisticRegression(C=0.3, class_weight='balanced', l1_ratio=0.2, max_iter=5000,
                   penalty='elasticnet', solver='saga')

In [6]:
pipeline = Pipeline(steps=[
    ('Applying transformation & preprossing', preprocessor),
    ('Model Tarining', LogisticRegression())]
)
pipeline.fit(X_train,y_train)
y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

0.6552642857142857


### Using XGBoost

In [7]:
xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    eval_metric='auc',
    n_jobs=-1,
    random_state=42
)

pipeline = Pipeline(steps=[
    ('Applying transformation & preprossing', preprocessor),
    ('Model Tarining', xgb_model)]
)

pipeline.fit(X_train,y_train)
y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

0.6837142857142857


In [ ]:
y_pred_test = pipeline.predict(df_test)

In [81]:
submission = pd.DataFrame({
    'id': df_test['id'],
    'diagnosed_diabetes': y_pred_test
})

In [82]:
submission.sample(10)

,id,diagnosed_diabetes
191744,891744,0.451948
97561,797561,0.440520
86062,786062,0.750661
205917,905917,0.681813
76748,776748,0.552379
126383,826383,0.350190
203021,903021,0.842355
11338,711338,0.755257
65226,765226,0.582492
145467,845467,0.399789


In [83]:
submission.to_csv("submission.csv", index=False)